<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/5%EC%A3%BC%EC%B0%A8_%EB%8F%84%EC%A0%84%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 🏆 STEP 9. 도전 과제 — 리뷰 추가 후 재학습

**목표:** 직접 리뷰를 추가해서 재학습하고 유사도 변화를 확인합니다.

> 💡 **현실 팁**
> Word2Vec은 데이터가 많을수록 품질이 올라갑니다.
> "연기"와 "배우" 유사도가 0.95 이상이면 학습이 잘 된 것입니다.

### 🔧 준비: 기존 모델 학습

> ⬇️ 셀을 **순서대로** 실행하세요 (Shift+Enter)

In [ ]:
# 필요한 도구 설치
!pip install gensim --quiet
print("✅ 완료!")

In [ ]:
# 필요한 도구 불러오기
from gensim.models import Word2Vec
import numpy as np

print("✅ 준비 완료!")

In [ ]:
# 기존 학습 데이터 — 영화 '왕과 사는 남자' 관람 리뷰 (29개)
문장들 = [
    # ─── 연기 (긍정) ────────────────────────────
    ["주인공", "연기", "훌륭하다", "감동", "자연스럽다"],
    ["배우", "연기력", "뛰어나다", "캐릭터", "몰입"],
    ["주연", "조연", "연기", "앙상블", "완벽하다"],
    ["배우", "표정", "눈빛", "감정", "전달"],
    ["주인공", "카리스마", "연기", "압도적", "최고"],
    ["배우", "연기", "눈빛", "섬세하다", "감탄"],

    # ─── 연기 (부정) ────────────────────────────
    ["배우", "연기", "어색하다", "어색", "실망"],
    ["주인공", "연기력", "부족하다", "아쉽다"],

    # ─── 스토리 (긍정) ──────────────────────────
    ["스토리", "탄탄하다", "전개", "자연스럽다", "몰입"],
    ["각본", "훌륭하다", "반전", "감동", "여운"],
    ["이야기", "감동적", "눈물", "여운", "오래"],
    ["스토리", "감동", "결말", "인상적", "훌륭하다"],

    # ─── 스토리 (부정) ──────────────────────────
    ["스토리", "지루하다", "전개", "느리다", "졸렸다"],
    ["각본", "허술하다", "개연성", "부족하다", "실망"],
    ["이야기", "예측", "뻔하다", "지루하다"],

    # ─── 영상미 / 음악 (긍정) ───────────────────
    ["영상미", "아름답다", "촬영", "연출", "훌륭하다"],
    ["OST", "음악", "감동적", "여운", "좋다"],
    ["배경", "아름답다", "영상", "색감", "완벽하다"],
    ["연출", "감독", "세밀하다", "장면", "인상적"],

    # ─── 영상미 (부정) ──────────────────────────
    ["CG", "조잡하다", "특수효과", "실망", "별로"],

    # ─── 왕과 사는 남자 특화 ────────────────────
    ["왕", "역사", "고증", "완벽하다", "배경"],
    ["왕", "신하", "관계", "감동", "이야기"],
    ["궁궐", "의상", "아름답다", "볼거리", "훌륭하다"],
    ["왕", "배우", "연기", "압도적", "최고"],
    ["역사", "드라마", "감동", "눈물", "여운"],

    # ─── 전반 긍정 ──────────────────────────────
    ["최고", "영화", "올해", "추천", "감동"],
    ["재관람", "추천", "여운", "오래", "훌륭하다"],

    # ─── 전반 부정 ──────────────────────────────
    ["실망", "기대", "별로", "돈", "아깝다"],
    ["최악", "지루하다", "졸렸다", "실망", "별로"],
]

print(f"기존 학습 문장 수: {len(문장들)}개")

In [ ]:
# 기존 모델 학습
model = Word2Vec(
    sentences=문장들,
    vector_size=30, window=4, min_count=1, epochs=500, seed=42
)

print("✅ 기존 모델 학습 완료!")
print(f"   어휘 크기: {len(model.wv)}개 단어")
print()

# 기존 모델 유사도 확인
print("기존 모델 유사도:")
print("-" * 42)
기존_쌍 = [("연기","배우"), ("감동","여운"), ("지루하다","별로"),
          ("왕","역사"), ("영상미","아름답다")]
for a, b in 기존_쌍:
    sim = model.wv.similarity(a, b)
    막대 = "█" * int(sim * 20)
    print(f"  '{a}'↔'{b}': {sim:.4f}  {막대}")

---
### 리뷰 추가 — '왕과 사는 남자' 리뷰 100개로 확장

기존 29개 문장에 **71개를 추가**하여 총 100개로 재학습합니다.
데이터가 늘어날수록 유사도가 더 안정적으로 나오는지 확인합니다.

In [ ]:
# 리뷰 추가 — 71개 추가하여 총 100개
추가_문장들 = [
    # ─── 연기 (긍정) 추가 14개 ──────────────────
    ["배우", "연기", "완벽하다", "눈물", "감동"],
    ["주연", "연기력", "최고", "카리스마", "인상적"],
    ["배우", "감정", "표현", "자연스럽다", "몰입"],
    ["주인공", "눈빛", "연기", "소름", "대단하다"],
    ["배우", "캐릭터", "몰입", "완벽하다", "훌륭하다"],
    ["주연", "조연", "케미", "환상적", "감탄"],
    ["광대", "연기", "코믹", "찰떡", "웃음"],
    ["왕", "광대", "1인2역", "연기", "소름"],
    ["배우", "내공", "눈빛", "전달", "감동"],
    ["사극", "연기", "자연스럽다", "담백하다", "좋다"],
    ["배우", "대사", "감정", "전달", "감동적"],
    ["주인공", "변화", "연기", "압권", "최고"],
    ["표정", "연기", "명장면", "인상적", "훌륭하다"],
    ["배우", "믿고보다", "확인", "연기력", "최고"],

    # ─── 연기 (부정) 추가 6개 ───────────────────
    ["연기", "과장", "몰입", "안되다", "실망"],
    ["사투리", "연기", "어색하다", "집중", "안되다"],
    ["조연", "연기력", "부족하다", "아쉽다", "별로"],
    ["감정", "표현", "단조롭다", "지루하다", "실망"],
    ["대사", "전달", "부자연스럽다", "어색하다", "별로"],
    ["눈물", "연기", "억지스럽다", "감동", "안되다"],

    # ─── 스토리 (긍정) 추가 10개 ─────────────────
    ["왕", "광대", "운명", "설정", "신선하다"],
    ["권력", "인간성", "메시지", "인상적", "감동"],
    ["왕", "외로움", "이야기", "가슴", "여운"],
    ["웃음", "감동", "각본", "완벽하다", "훌륭하다"],
    ["광대", "왕", "과정", "설득력", "몰입"],
    ["사극", "현대적", "메시지", "좋다", "인상적"],
    ["전개", "자연스럽다", "몰입감", "뛰어나다", "최고"],
    ["왕", "신하", "관계", "흥미진진하다", "재미"],
    ["반전", "이야기", "깊다", "감동", "여운"],
    ["코미디", "드라마", "구성", "좋다", "재미"],

    # ─── 스토리 (부정) 추가 8개 ──────────────────
    ["전개", "느리다", "졸렸다", "지루하다", "실망"],
    ["뻔하다", "예측", "긴장감", "없다", "별로"],
    ["설정", "비현실적", "개연성", "부족하다", "아쉽다"],
    ["결말", "급하다", "마무리", "아쉽다", "실망"],
    ["중반", "늘어지다", "집중", "안되다", "지루하다"],
    ["톤", "전환", "어색하다", "코미디", "별로"],
    ["역사", "왜곡", "불편하다", "실망", "별로"],
    ["이야기", "산만하다", "정리", "안되다", "아쉽다"],

    # ─── 영상미 / 음악 (긍정) 추가 8개 ──────────
    ["궁궐", "촬영", "영상미", "아름답다", "감탄"],
    ["의상", "세트", "화려하다", "완벽하다", "볼거리"],
    ["OST", "감동", "장면", "배로", "좋다"],
    ["촬영", "구도", "그림", "아름답다", "인상적"],
    ["야경", "촬영", "장면", "인상적", "아름답다"],
    ["색감", "따뜻하다", "눈", "편안하다", "좋다"],
    ["음악", "감정선", "맞아떨어지다", "완벽하다", "감동"],
    ["의상", "캐릭터", "표현", "디자인", "훌륭하다"],

    # ─── 영상미 (부정) 추가 4개 ──────────────────
    ["CG", "티나다", "몰입", "깨지다", "별로"],
    ["세트", "조잡하다", "분위기", "안나다", "실망"],
    ["조명", "어둡다", "화면", "안보이다", "별로"],
    ["특수효과", "수준", "낮다", "실망", "아깝다"],

    # ─── 왕과 사는 남자 특화 추가 8개 ────────────
    ["왕", "카리스마", "광대", "순수함", "대비"],
    ["궁궐", "생활", "왕", "고독", "감동"],
    ["광대", "백성", "마음", "왕", "변화"],
    ["신하", "충성", "배신", "긴장감", "이야기"],
    ["왕", "광대", "바뀌다", "운명", "드라마"],
    ["역사", "배경", "조선", "사극", "재미"],
    ["왕", "권력", "외로움", "눈물", "감동"],
    ["광대", "지혜", "용기", "감동", "여운"],

    # ─── 전반 긍정 추가 7개 ──────────────────────
    ["올해", "최고", "추천", "감동", "재미"],
    ["재관람", "가치", "있다", "여운", "감동"],
    ["가족", "같이", "보다", "좋다", "추천"],
    ["명작", "남녀노소", "즐기다", "감동", "최고"],
    ["두번째", "감동", "줄지않다", "여운", "최고"],
    ["한국", "사극", "자존심", "작품", "훌륭하다"],
    ["극장", "꼭보다", "추천", "강력", "최고"],

    # ─── 전반 부정 추가 6개 ──────────────────────
    ["기대", "실망", "크다", "돈", "아깝다"],
    ["시간", "길다", "지루하다", "낭비", "별로"],
    ["평점", "높다", "왜", "모르겠다", "실망"],
    ["최악", "사극", "다시", "안보다", "별로"],
    ["홍보", "거창하다", "내용", "별로", "실망"],
    ["환불", "하고싶다", "실망", "아깝다", "별로"],
]

확장_문장들 = 문장들 + 추가_문장들
print(f"기존 {len(문장들)}개 → 확장 {len(확장_문장들)}개")

In [ ]:
# 재학습
model_v2 = Word2Vec(
    sentences=확장_문장들,
    vector_size=30, window=4, min_count=1, epochs=500, seed=42
)

print("유사도 변화 비교:")
print(f"{'단어 쌍':22}  {'기존':8}  {'재학습후'}")
print("-" * 42)
비교쌍 = [("연기","배우"), ("감동","여운"), ("지루하다","별로"),
          ("왕","역사"), ("영상미","아름답다"), ("실망","아깝다"),
          ("스토리","이야기"), ("광대","왕"), ("추천","최고")]
for a, b in 비교쌍:
    이전 = model.wv.similarity(a, b)
    이후 = model_v2.wv.similarity(a, b)
    변화 = "↑" if 이후 > 이전 else "↓" if 이후 < 이전 else "="
    print(f"  '{a}'↔'{b}':  {이전:.4f}   {이후:.4f}  {변화}")

In [ ]:
# 유사 단어 검색 비교
검색_단어들 = ["왕", "연기", "감동", "실망", "영상미", "광대"]

print("유사 단어 검색 비교:")
print("=" * 60)
for 단어 in 검색_단어들:
    기존_유사어 = model.wv.most_similar(단어, topn=4)
    재학습_유사어 = model_v2.wv.most_similar(단어, topn=4)
    기존_top = [f"'{w}'({s:.2f})" for w, s in 기존_유사어]
    재학습_top = [f"'{w}'({s:.2f})" for w, s in 재학습_유사어]
    print(f"  '{단어}' 유사어:")
    print(f"    기존:   {', '.join(기존_top)}")
    print(f"    재학습: {', '.join(재학습_top)}")
    print()

In [ ]:
# ✏️ 여기에 직접 리뷰 단어 목록을 추가해서 다시 학습해보세요!
나의_추가_문장들 = [
    # ["단어1", "단어2", "단어3", "단어4", "단어5"],
    # ["단어1", "단어2", "단어3", "단어4", "단어5"],
]

if len(나의_추가_문장들) > 0:
    나의_확장_문장들 = 확장_문장들 + 나의_추가_문장들
    model_v3 = Word2Vec(
        sentences=나의_확장_문장들,
        vector_size=30, window=4, min_count=1, epochs=500, seed=42
    )
    print(f"나의 데이터 추가: {len(확장_문장들)}개 → {len(나의_확장_문장들)}개")
    print()
    print("유사도 변화:")
    print(f"{'단어 쌍':22}  {'100개':8}  {'추가후'}")
    print("-" * 42)
    for a, b in [("연기","배우"), ("감동","여운"), ("지루하다","별로")]:
        이전 = model_v2.wv.similarity(a, b)
        이후 = model_v3.wv.similarity(a, b)
        변화 = "↑" if 이후 > 이전 else "↓"
        print(f"  '{a}'↔'{b}':  {이전:.4f}   {이후:.4f}  {변화}")
else:
    print("⬆️ 위에 나의_추가_문장들에 직접 리뷰를 추가한 후 다시 실행하세요!")

> 💡 **현실 팁**
>
> 유사도가 0.95 이상으로 안정적으로 나오기 시작하면 학습이 잘 된 것입니다.
>
> 반대로 원하지 않는 단어가 유사어로 나온다면, 해당 단어들이 함께 등장하는 문장이 많다는 뜻입니다.
>
> 데이터를 보강하거나 불필요한 문장을 제거해서 해결합니다.